<a href="https://colab.research.google.com/github/bettercalln1ck/2018-CFI-CTF/blob/master/Gaming_candidate_evaluation_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaming — Candidate Evaluation Notebook

**Goal:** Design **one** Gaming task that tests whether an LLM can **learn the rules of a game from context** — and provably **fails without them**.

> 📤 **How to submit:** Make a **copy** of this notebook, fill in every section with your task, and submit the completed `.ipynb` file. Do not modify the original template.

---

## ✅ What you must complete

1. **Fill in task metadata:** `Category: Gaming`, `Sub-category: Synthetic Game` or `Recent Board Game`, and `Topic`.
2. **Write a realistic game prompt** that demands a concrete JSON answer and is unsolvable without context.
3. **Provide the Gold Context** — the complete, authoritative rule documentation.
4. **Provide the Golden Answer.**
5. **Implement the deterministic checker** `check_prediction(pred, expected) -> float`.
6. **Sign off the Quality Checklist.**


## 1) Task Metadata

Use exactly these metadata fields. Set `Sub-category` to either `Synthetic Game` or `Recent Board Game`.


```json
{
    "Category": "Gaming",
    "Sub-category": "Synthetic Game",
    "Topic": "<Your Game Title>"
}
```


## 2) Write the Prompt

Write a **realistic, decision-forcing game prompt**:

- **Self-contained:** Includes the complete starting state (grid, hand, board snapshot, scores, deck size).
- **Decision-forcing:** Asks for a specific output — a move sequence, a turn plan, the resolved end-state — not commentary.
- **Deterministic:** Given the rules and state, **exactly one** answer is optimal.
- **Reasoning-free output:** The model returns structured JSON only — no chain-of-thought.
- **Structured output schema:** Specify the JSON shape explicitly; instruct the model to return a single JSON object inside a fenced ```json``` block, with no text outside it.

The prompt must be **unsolvable without** the Gold Context you will write next.

> 💡 The grid example below is illustrative. Card games, dice games, resource-management mini-games, or any other game format are equally valid — adapt the prompt to whatever state your game uses (hand of cards, deck remaining, board layout, etc.).


````python
# === Candidate Input: Prompt ===
prompt_text = """You are an advanced strategy engine for a grid-based puzzle game.

GAME STATE (10x10 grid; row 0 = top, col 0 = left):
grid = [
  ["O", 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
  [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
  [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
  [0, 1, 0, 2, 2, 0, 1, 0, 1, 0],
  [0, 1, 0, 2, 0, 0, 1, 0, 1, 0],
  [0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
  [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, "E"],
]

TASK
Plan a sequence of moves that takes the player from `E` to `O` without dying.
Return the move list as JSON.

REQUIRED OUTPUT FORMAT
Return ONLY a single JSON object inside a fenced ```json``` block. No text
outside the block, no commentary, no chain-of-thought.

```json
{"moves": [["direction", value], ["direction", value], ...]}
```
"""
print(prompt_text)
````


## 3) Provide the Gold Context

The **Gold Context** is the heart of your task — the complete, authoritative rule documentation needed to solve your prompt.

It must be:

- **Self-contained** — every symbol, legend entry, and rule the prompt depends on is defined here; no external references.
- **Comprehensive** — covers all rules the model needs to reach the correct answer.
- **Internally consistent** — no contradictions; produces one unique correct answer.
- **Well-structured** — clear numbered sections (Environment, Legend, Movement Rules, Mechanics, Worked Example).
- **Anchored with one tiny worked example** — a small instance and its solved answer, so the model can pattern-match the format.

A careful human reader should be able to solve the prompt using **only** the Gold Context. A frontier model **should not** be able to solve the prompt without it.


> **Game Documentation: &lt;Your Game Name&gt;**
>
> **1. The Environment** — [Describe the grid / board / play area dimensions and coordinate system.]
>
> **2. The Legend**
> - `0` (Empty): safe / free movement
> - `1` (Wall): instant death — cannot land on or pass through
> - `2` (Snake): trap — landing on it teleports you to ...
> - `E` (Entry): starting tile
> - `O` (Out/Exit): winning tile
>
> **3. Movement Rules**
> - Input format: `["Direction", Value]`
> - Valid directions: `U` (north), `D` (south), `L` (west), `R` (east)
> - Each move attempts to jump `Value` squares in that direction.
>
> **4. The Dice/Resource Mechanic** — [Describe any pool / hand / resource constraint that makes the puzzle non-trivial.]
>
> **5. Worked Example**
>
> ```
> mini_grid = [
>   ["O", 0, 1, 0, 0],
>   [ 0, 0, 0, 0, 0],
>   [ 2, 2, 1, 0, 0],
>   [ 0, 0, 0, 0, 0],
>   [ 0, 0, 1, 0, "E"]
> ]
> ```
>
> Solution from `E`: `["U", 3]` → `["L", 4]` → `["U", 1]`
>
> Final answer: `[["U", 3], ["L", 4], ["U", 1]]`

*Replace the blockquoted template above with your full game documentation.*


## 4) Golden Answer (Response)

Paste the **golden JSON answer** the model is expected to produce, inside a fenced ```json``` block. This is the single ground truth — the validator will compare the model's output against it.


```json
{"moves": [["U", 6], ["U", 3], ["L", 5], ["L", 4]]}
```


## 5) Implement the Checker

Your `check_prediction(pred, expected)` function must:

- Return a **float in `[0.0, 1.0]`**.
- Be **deterministic** (same inputs → same output).
- Be **self-contained** — embed any grid/board/state constants the checker needs.
- Use only **stdlib** imports (`json`, `re`, `ast`, `math`).
- Compare against the **single golden answer** (`EXPECTED`). No rubric, no equivalence classes, no alternative valid paths.

Use **test-style scoring**: write a list of discrete pass/fail checks against the golden answer and return `passing_tests / total_tests`. Add as many tests as you have fields. For a single-field schema, the list has one test and the score is `1.0` on match, `0.0` otherwise.


In [ ]:
# === Candidate Input: Validator ===
# Embed the golden answer from section 4. The validator runs a list
# of discrete checks (tests) against EXPECTED and returns
# passing_tests / total_tests. No rubric, no equivalence classes,
# no alternative valid paths.

EXPECTED = {
    "moves": [["U", 6], ["U", 3], ["L", 5], ["L", 4]],
}


def check_prediction(pred, expected=EXPECTED) -> float:
    """
    Test-style scoring against the single golden EXPECTED answer.
      score = passing_tests / total_tests
        1.0 -> every test passes
        0.0 -> malformed input or no test passes
    """
    if not isinstance(pred, dict):
        return 0.0

    # Add one assertion per field in EXPECTED. Each line is a single test
    # that compares the prediction against the golden answer.
    tests = [
        pred.get("moves") == expected.get("moves"),
    ]

    total = len(tests)
    passing = sum(1 for t in tests if t)
    return passing / total if total else 0.0


## 6) ✅ Quality Checklist

Before submission, confirm **every** box. If any is unchecked, revise the task.

- [ ] **Context-Dependent** — Confirmed with a frontier model that the prompt alone (no Gold Context) produces 0% success.
- [ ] **Novel** — Rules are invented (`Synthetic Game`) or the game is niche/recent enough that models can't recite it.
- [ ] **No Banned Games** — See the banned-games list in the instructions doc.
- [ ] **Self-Contained Gold Context** — All rules, legends, examples are in the Gold Context cell.
- [ ] **Single Golden Answer** — There is exactly one ground-truth answer. Partial credit is graded against it; no alternative-path or equivalence-class acceptance.
- [ ] **Structured Output** — Prompt forces a single JSON object inside a fenced ```json``` block, with no surrounding text.
- [ ] **Automated Checker** — Runs without manual input, returns a float in `[0.0, 1.0]`.
- [ ] **Reasoning-Free Output** — Prompt asks for JSON only — no narrative inside the answer block.

When all boxes are checked ✅, your task is ready for submission.
